# Phone → text transcription with a fine-tuned pretrained seq2seq

The CTC acoustic model (`src/ctc/`) stays exactly as built — a from-scratch **phone classifier**.
On top of its phone output we add a **pretrained sequence-to-sequence model**, lightly fine-tuned to
translate a phone string into orthographic Polish text:

```
wav → CTCModel → greedy phones → "sil b r u n o sil ..." → seq2seq → "Bruno, ..."
```

Why this works for ~20 h of data: we don't train the text model from scratch — we fine-tune
`allegro/plt5-small` (Polish T5 from Hugging Face), whose Polish pretraining acts as a built-in
language model. The seq2seq learns spelling, word segmentation, **and** how to correct the acoustic
model's systematic phone errors — because it is trained on the CTC model's *predicted* phones, not
oracle alignments. Everything runs on Windows (`transformers` + `torch`, no KenLM/Flashlight).

Evaluation is on a **speaker-disjoint** test split and reported as **WER / CER**, compared against a
no-language-model heuristic baseline (`wordmaker.phonemes_to_text` + sil-based word splitting).

## Setup

The dataset (CTC-predicted phones + transcripts) and the fine-tuned model are both cached, so the
first run does the heavy work and later runs are fast. Set `FORCE_RETRAIN = True` to retrain.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent

import torch
import matplotlib.pyplot as plt

from src.p2g.data import build_pairs, split_by_speaker, read_jsonl, write_jsonl
from src.p2g.model import P2GModel
from src.p2g.train import train_p2g, evaluate
from src.ctc.inference import load_ctc_model
from src.p2g.transcribe import transcribe_wav
from src.p2g.metrics import corpus_wer, corpus_cer
from src.wordmaker import phonemes_to_text
from src.utils.device import get_device

# --- Config -----------------------------------------------------------------
CTC_CHECKPOINT = ROOT / "trained_models" / "ctc_all_augmentations_45epochs.pt"
MODEL_NAME = "allegro/plt5-small"            # Polish T5; fallback: google/byt5-small
DATA_DIR = ROOT / "data"
MAX_FILES = None                               # sampled across speakers (set None for all)
EPOCHS = 5
BATCH_SIZE = 8
NUM_BEAMS = 4
LR = 3e-4                                      # AdamW peak LR (warmup -> linear decay inside train_p2g)
EARLY_STOP_PATIENCE = 3                        # stop if val WER stalls for N epochs (None disables)
DATA_CACHE = ROOT / "data" / "p2g"            # train/val/test.jsonl cache
SAVE_DIR = ROOT / "trained_models" / "p2g_plt5_small"
FORCE_RETRAIN = False
DEVICE = get_device()
print("device:", DEVICE)

## 1. Build (or load) the (predicted-phones → text) dataset

For each utterance we run the trained CTC model over the audio, greedy-decode to a phone string
(keeping `sil` as word-boundary cues), and pair it with the `.txt` transcript. The split is
speaker-disjoint. This is cached to JSONL on first run.

In [ ]:
splits = {s: DATA_CACHE / f"{s}.jsonl" for s in ("train", "val", "test")}
if all(p.exists() for p in splits.values()):
    train_rows = read_jsonl(splits["train"])
    val_rows = read_jsonl(splits["val"])
    test_rows = read_jsonl(splits["test"])
    print("loaded cached dataset")
else:
    print("building dataset with CTC-predicted phones (one-time)...")
    rows = build_pairs(
        DATA_DIR, mode="pred", checkpoint=str(CTC_CHECKPOINT), device=DEVICE,
        max_files=MAX_FILES, shuffle=True,
    )
    train_rows, val_rows, test_rows = split_by_speaker(rows)
    write_jsonl(splits["train"], train_rows)
    write_jsonl(splits["val"], val_rows)
    write_jsonl(splits["test"], test_rows)

print(f"train {len(train_rows)} / val {len(val_rows)} / test {len(test_rows)} utterances")
print("\nexample pair:")
print("  phones:", train_rows[0]["phones"][:90], "...")
print("  text:  ", train_rows[0]["text"][:90])

## 2. No-LM heuristic baseline

The simplest phones→text rule: split the phone string on silence into words and map each phone to
Polish letters with `wordmaker.phonemes_to_text`. No language model, no spelling correction — this is
what the seq2seq has to beat.

In [ ]:
def heuristic_text(phone_string):
    """Split phones on silence into words, map each to letters, join with spaces."""
    words, cur = [], []
    for ph in phone_string.split():
        if ph in ("sil", "sp"):
            if cur:
                words.append(phonemes_to_text(cur, after_silence=True))
                cur = []
        else:
            cur.append(ph)
    if cur:
        words.append(phonemes_to_text(cur, after_silence=True))
    return " ".join(w for w in words if w)


base_preds = [heuristic_text(r["phones"]) for r in test_rows]
test_refs = [r["text"] for r in test_rows]
base_wer = corpus_wer(base_preds, test_refs)
base_cer = corpus_cer(base_preds, test_refs)
print(f"heuristic baseline  WER {base_wer:.3f}  CER {base_cer:.3f}")
print("\nexample:")
print("  ref :", test_refs[0][:90])
print("  base:", base_preds[0][:90])

## 3. Fine-tune (or load) the P2G seq2seq

Fine-tunes `allegro/plt5-small` on the predicted-phones→text pairs and saves it. Cached: later runs
just load the saved model unless `FORCE_RETRAIN`.

In [ ]:
if SAVE_DIR.exists() and not FORCE_RETRAIN:
    print("loading fine-tuned P2G model from", SAVE_DIR)
    p2g = P2GModel.from_pretrained(str(SAVE_DIR), device=DEVICE)
else:
    p2g, _ = train_p2g(
        train_rows, val_rows,
        model_name=MODEL_NAME, output_dir=str(SAVE_DIR),
        epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR, num_beams=NUM_BEAMS,
        early_stopping_patience=EARLY_STOP_PATIENCE, device=DEVICE,
    )

## 4. Evaluate on the speaker-disjoint test set

In [ ]:
res = evaluate(p2g, test_rows, batch_size=16, num_beams=NUM_BEAMS)
p2g_preds = res["preds"]
print(f"P2G (plt5-small)   WER {res['wer']:.3f}  CER {res['cer']:.3f}")
print(f"heuristic baseline WER {base_wer:.3f}  CER {base_cer:.3f}")

In [ ]:
# Example transcriptions: reference vs heuristic vs P2G.
for r, base, p in list(zip(test_refs, base_preds, p2g_preds))[:12]:
    print("REF :", r[:100])
    print("BASE:", base[:100])
    print("P2G :", p[:100])
    print("-" * 80)

## 5. End-to-end from a wav file

The full pipeline on raw audio: `wav → CTC phones → P2G text`.

In [ ]:
from src.p2g.data import discover_triples

ctc_model = load_ctc_model(str(CTC_CHECKPOINT), DEVICE)
wav, _, txt, _ = discover_triples(DATA_DIR)[0]
print("wav      :", wav.name)
print("reference:", Path(txt).read_text(encoding="utf-8").strip()[:120])
print("predicted:", transcribe_wav(str(wav), ctc_model, p2g, DEVICE, num_beams=NUM_BEAMS)[:120])

## 6. WER comparison

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
names = ["heuristic\n(no LM)", "P2G\n(plt5-small)"]
vals = [base_wer, res["wer"]]
bars = ax.bar(names, vals, color=["#9aa0a6", "#1a73e8"])
ax.set_ylabel("WER (lower is better)")
ax.set_title(f"Word error rate on {len(test_rows)} held-out utterances")
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.01, f"{v:.2f}", ha="center")
plt.tight_layout()
plt.show()

## Takeaways

* The from-scratch CTC phone model is unchanged; the pretrained seq2seq supplies the spelling, word
  segmentation and language modelling the phone classifier lacks.
* Training on the CTC model's *predicted* phones makes the seq2seq an error-correcting decoder, so it
  recovers from acoustic mistakes a clean-phone transliterator could not.
* This is open-vocabulary and Windows-only (no KenLM/Flashlight/WSL).
* To push further: more fine-tuning data/epochs, try `allegro/plt5-base`, or feed n-best CTC phone
  hypotheses instead of the single greedy decode.